In [ ]:
# in this notebook we read and combine two datasets
# hydroportal gives site codes and their corresponding states and locations
# usda gives timeseries data for each site and state code

In [20]:
import ulmo

In [25]:
import os
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import datetime

In [19]:
# first import the list of sites which gives their numerical codes, state, and location (longitude and latitude)
# read data from hydroportal site
# formate dataframe
#wsdlurl = 'https://hydroportal.cuahsi.org/Snotel/cuahsi_1_1.asmx?WSDL'
#sites_hydro = ulmo.cuahsi.wof.get_sites(wsdlurl)
#data_hydro = pd.DataFrame.from_dict(sites_hydro, orient='index').dropna()
#df_hydro = data_hydro.drop(columns=['network','elevation_m','site_property']).rename(columns={'code': 'site'})
#codes_hydro = df_hydro['site'].str.split('_', expand=True)
#df_hydro['site'] = df_hydro['site'].str.extract(r'(\d+)).astype(int).astype(str)

In [99]:
file_hydro = "/Users/lizag/Downloads/nwcc_inventory.csv"
data_hydro = pd.read_csv(file_hydro, sep = ',')
df_hydro = data_hydro.drop(columns=['network','start_date','end_date','element','county','Subwatershed','site_info_link',
                                   'cdbs_id', 'shef_id','gmt_offset']).rename(columns={'station id': 'site'})

In [122]:
df_hydro.head()

,site,lat,lon,elev,state,site_name
0,1267,61.75,-150.89,170,AK,Alexander Lake
1,1189,64.79,-141.23,1020,AK,American Creek
2,1062,59.86,-151.31,1650,AK,Anchor River Divide
3,1070,61.11,-149.68,1910,AK,Anchorage Hillside
4,2065,61.58,-159.58,90,AK,Aniak


In [121]:
sites = df_hydro['site'].astype(str)
states = df_hydro['state'].astype(str)
names = df_hydro['site_name'].astype(str)

In [103]:
#read timeseries data from usda site
# returns data, SWE over date range
def read_timeseries_data(site_num,state):
    
    #website_name = 'https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/customMultiTimeSeriesGroupByStationReport/daily/start_of_period/'+site_num+':'+state+':SNTL%257Cid=%2522%2522%257Cname/POR_BEGIN,POR_END/WTEQ::value,WTEQ::collectionDate,name,stationId,latitude,longitude?fitToScreen=true'
    #website_name = 'https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/customMultiTimeSeriesGroupByStationReport/daily/start_of_period/'+site_num+':'+state+':SNTL%257Cid=%2522%2522%257Cname/POR_BEGIN,POR_END/WTEQ::value?fitToScreen=true'
    website_name = 'https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/customMultiTimeSeriesGroupByStationReport/daily/start_of_period/'+site_num+':'+state+':SNTL%257Cid=%2522%2522%257Cname/POR_BEGIN,POR_END/WTEQ::value,SNWD::value,PREC::value,PRCP::value,PRCPMTD::value,TAVG::value,TMAX::value,TMIN::value,TOBS::value,PRCPSA::value,SNDN::value,SNRR::value,WTEQX::value?fitToScreen=true'
    #try:
    #    data_usda = pd.read_csv(website_name, comment='#')
    data_usda = pd.read_csv(website_name, comment='#')
    #except (http.client.IncompleteRead) as e:
    #    data_usda = e.partial
    return data_usda

In [144]:
# remove sites in list without data
def determine_site_has_data(site):
    vacants = [188, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 261, 262, 263, 264, 266, 267, 268, 269, 372, 397, 421, 431, 
               441, 498, 598, 611, 630, 659, 678, 685, 758, 799, 851, 976, 994, 995, 996, 1004, 1007, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 
               1026, 1027, 1028, 1029, 1049, 1157, 1276, 8011, 8082,302,112]
    if site in vacants:
        return False
    else:
        return True

In [145]:
# renames column titles
# adds site number to each data and SWE entry
# remove dates before April 3, 2014
def format_timeseries_data(data_usda,site_num,state,name):    
    #column1 = name +' ('+site_num+') Snow Water Equivalent (in) Start of Day Values'
    column1 = list(data_usda)[1]
    column2 = list(data_usda)[2]
    column3 = list(data_usda)[3]
    column4 = list(data_usda)[4]
    column5 = list(data_usda)[5]
    column6 = list(data_usda)[6]
    column7 = list(data_usda)[7]
    column8 = list(data_usda)[8]
    column9 = list(data_usda)[9]
    column10 = list(data_usda)[10]
    column11 = list(data_usda)[11]
    column12 = list(data_usda)[12]
    column13 = list(data_usda)[13]
    #column2 = name + ' ('+site_num+') Snow Water Equivalent Collection Date Start of Day Values'
    df_usda = data_usda.rename(columns={column1: 'SWE (in)',column2: 'Depth SoD (in)', column3: 'Precip Accum SoD (in)', column4: 'Precip Inc (in)',
                                       column5: 'Precip Month-to-Date SoD (in)', column6: 'Air Temp Avg (F)', column7: 'Air Temp Max (F)', 
                                        column8: 'Air Temp Min (F)', column9: 'Air Temp Obs SoD (F)', column10: 'Precip Inc Snow-adj (in)', 
                                       column11: 'Snow Density SoD (pct)', column12: 'Snow Rain Ratio', column13: 'SWE Max (in)'})#.drop(columns = column2)

    #snotel_site = int(''.join(filter(str.isdigit, data_usda.columns[1])))
    df_usda['site'] = str(site_num) #snotel_site
    #df_usda=df_usda.set_index('site')

    df_usda["Date"]=pd.to_datetime(df_usda["Date"])
    df_usda=df_usda[~(df_usda['Date'] < '2014-04-03')]
    return df_usda

In [146]:
data_usda = read_timeseries_data(str(sites[111]),str(states[111]))
df_usda = format_timeseries_data(data_usda,str(sites[111]),str(states[111]),str(names[111]))

In [147]:
data_usda.tail()

,Date,Poison Flat (697) Snow Water Equivalent (in) Start of Day Values,Poison Flat (697) Snow Depth (in) Start of Day Values,Poison Flat (697) Precipitation Accumulation (in) Start of Day Values,Poison Flat (697) Precipitation Increment (in),Poison Flat (697) Precipitation Month-to-date (in) Start of Day Values,Poison Flat (697) Air Temperature Average (degF),Poison Flat (697) Air Temperature Maximum (degF),Poison Flat (697) Air Temperature Minimum (degF),Poison Flat (697) Air Temperature Observed (degF) Start of Day Values,Poison Flat (697) Precipitation Increment - Snow-adj (in),Poison Flat (697) Snow Density (pct) Start of Day Values,Poison Flat (697) Snow Rain Ratio (unitless),Poison Flat (697) Snow Water Equivalent Maximum (in)
16270,2025-04-18,13.9,30.0,23.2,0.0,0.6,33.4,43.0,23.0,28.8,0.0,46.3,NaN,NaN
16271,2025-04-19,13.7,30.0,23.2,0.0,0.6,38.5,55.8,21.7,26.4,0.0,45.7,NaN,NaN
16272,2025-04-20,12.9,30.0,23.2,0.0,0.6,41.9,56.5,28.8,31.8,0.0,43.0,NaN,NaN
16273,2025-04-21,12.3,29.0,23.2,0.0,0.6,42.4,57.2,28.9,31.5,0.0,42.4,NaN,NaN
16274,2025-04-22,11.6,27.0,23.2,NaN,0.6,NaN,NaN,NaN,33.3,NaN,43.0,NaN,NaN


In [148]:
df_hydro.tail()

,site,lat,lon,elev,state,site_name
883,859,41.00,-106.91,8950,WY,Whiskey Park
884,868,42.82,-110.84,8060,WY,Willow Creek
885,872,42.28,-105.58,7900,WY,Windy Peak
886,875,44.80,-109.66,7650,WY,Wolverine
887,878,43.93,-109.82,8340,WY,Younts Peak


In [149]:
# merge datasets by site number
def merge_datasets(df_usda, df_hydro):
    df_merged = pd.merge(df_usda.astype(str), df_hydro.astype(str), on='site')
    #merged_df = pd.DataFrame.join(df, sites_df_form, on='site')
    return df_merged

In [150]:
merged_df = merge_datasets(df_usda, df_hydro)
merged_df.head()

,Date,SWE (in),Depth SoD (in),Precip Accum SoD (in),Precip Inc (in),Precip Month-to-Date SoD (in),Air Temp Avg (F),Air Temp Max (F),Air Temp Min (F),Air Temp Obs SoD (F),Precip Inc Snow-adj (in),Snow Density SoD (pct),Snow Rain Ratio,SWE Max (in),site,lat,lon,elev,state,site_name
0,2014-04-03,8.9,29.0,12.4,0.0,0.2,30.2,43.3,14.4,25.0,0.0,30.7,nan,nan,697,38.51,-119.63,7730,CA,Poison Flat
1,2014-04-04,8.9,29.0,12.4,0.0,0.2,31.3,41.4,21.6,29.5,0.0,30.7,nan,nan,697,38.51,-119.63,7730,CA,Poison Flat
2,2014-04-05,8.9,28.0,12.4,0.0,0.2,33.3,44.2,21.9,22.3,0.0,31.8,nan,nan,697,38.51,-119.63,7730,CA,Poison Flat
3,2014-04-06,8.9,27.0,12.4,0.0,0.2,36.3,49.8,24.1,28.2,0.0,33.0,nan,nan,697,38.51,-119.63,7730,CA,Poison Flat
4,2014-04-07,8.8,26.0,12.4,0.0,0.2,41.0,60.3,25.5,29.8,0.0,33.8,nan,nan,697,38.51,-119.63,7730,CA,Poison Flat


In [151]:
from http.client import IncompleteRead

In [152]:
import http

In [172]:
df_final = pd.DataFrame()
for i in range(0, len(sites)):
    site_has_data = determine_site_has_data(int(sites[i]))
    if site_has_data == False:
        print('vacancy')
    #elif i == 112:
    #    print('we hate 112')
    #elif i == 118:
    #    print('we hate 118')
    #elif i == 120:
    #    print('we hate 120')
    #elif i == 139:
    #    print('we hate 139')
    elif "#" in names[i]:
        print('we hate #')
    elif site_has_data == True: 
        data_usda = read_timeseries_data(sites[i],states[i])
        df_usda = format_timeseries_data(data_usda,sites[i],states[i],names[i])
        #final_df_timeseries =  pd.concat([final_df, df_usda])
        merged_df = merge_datasets(df_usda, df_hydro)
        df_final = pd.concat([df_final, merged_df])
        #print(df.head())
        print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
vacancy
98
99
100
101
102
103
104
105
106
107
108
109
110
111
we hate #
113
114
115
116
117
we hate #
119
we hate #
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
we hate #
140
141
142
143
144
145
146
147
148
we hate #
vacancy
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267

In [174]:
states[i]

'WY'

In [167]:
i

149

In [175]:
filename = 'snotel_data_alldata_morecolumns.csv'
df_final.to_csv(filename, index=False)

In [ ]:
i

In [176]:
df_final.head()

,Date,SWE (in),Depth SoD (in),Precip Accum SoD (in),Precip Inc (in),Precip Month-to-Date SoD (in),Air Temp Avg (F),Air Temp Max (F),Air Temp Min (F),Air Temp Obs SoD (F),Precip Inc Snow-adj (in),Snow Density SoD (pct),Snow Rain Ratio,SWE Max (in),site,lat,lon,elev,state,site_name
0,2014-10-01,0.0,0.0,0.0,0.0,nan,nan,nan,nan,nan,0.0,nan,nan,nan,1267,61.75,-150.89,170,AK,Alexander Lake
1,2014-10-02,0.0,0.0,0.0,0.0,0.0,nan,nan,nan,nan,0.0,nan,nan,nan,1267,61.75,-150.89,170,AK,Alexander Lake
2,2014-10-03,0.0,0.0,0.0,0.0,0.0,nan,nan,nan,nan,0.0,nan,nan,nan,1267,61.75,-150.89,170,AK,Alexander Lake
3,2014-10-04,0.0,0.0,0.0,0.0,0.0,nan,nan,nan,nan,0.0,nan,nan,nan,1267,61.75,-150.89,170,AK,Alexander Lake
4,2014-10-05,0.0,0.0,0.0,0.0,0.0,37.4,43.7,32.7,nan,0.0,nan,nan,nan,1267,61.75,-150.89,170,AK,Alexander Lake
